In [1]:
import os, sys, re
import time

import random
import string

import subprocess

from PIL import Image
import pillow_avif


# Libraries for VAE
import torch
import numpy as np
from PIL import Image
from torchvision import transforms
from diffusers.models import AutoencoderKL
import zlib

# VAE

### -------------------------VAE settings ---------------------------------------

In [2]:
# Load VAE
vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse").eval()

# Compression config
TARGET_SIZE = (128, 128)  # Resize image to this before encoding
LATENT_CHANNELS = 4       # VAE outputs 4 channels at 1/8 size (so 16x16x4 for 128x128)

# Normalize utilities
normalize = transforms.Normalize([0.5], [0.5])
denormalize = transforms.Normalize([-1], [1 / 0.5])

In [3]:
#### -------------------------End VAE settings -----------------------------------

### Cmpress image using VAE

In [4]:
# ---------- Compress ----------
def compress_image_for_lora(image_path, output_bin):
    image = Image.open(image_path).convert("RGB")
    #image = Image.open(image_path)
    image = image.resize(TARGET_SIZE)
    tensor = transforms.ToTensor()(image)
    tensor = normalize(tensor).unsqueeze(0)

    with torch.no_grad():
        latent_dist = vae.encode(tensor)
        latent = latent_dist.latent_dist.sample() * 0.18215  # [1, 4, 16, 16]

    latent_np = latent.cpu().numpy()  # Shape: [1, 4, 16, 16]

    # Quantize to uint8 (0-255)
    latent_uint8 = ((latent_np + 2) / 4 * 255).clip(0, 255).astype(np.uint8)

    # Compress using zlib
    compressed = zlib.compress(latent_uint8.tobytes(), level=9)

    with open(output_bin, "wb") as f:
        f.write(compressed)

    #print(f"✅ Latent vector saved to {output_bin}, size = {len(compressed)} bytes")
    #print(sys.getsizeof(compressed))

### Decompress image using VAE

In [5]:
def decompress_image_from_lora(input_bin, output_img):
    with open(input_bin, "rb") as f:
        compressed = f.read()

    decompressed_bytes = zlib.decompress(compressed)
    latent_uint8 = np.frombuffer(decompressed_bytes, dtype=np.uint8)
    latent_np = latent_uint8.astype(np.float32) / 255 * 4 - 2  # Dequantize

    # Reshape to original latent shape: [1, 4, 16, 16]
    latent_tensor = torch.tensor(latent_np.reshape(1, LATENT_CHANNELS, 16, 16))

    with torch.no_grad():
        recon = vae.decode(latent_tensor / 0.18215).sample

    recon_img = recon.squeeze(0).cpu().clamp(-1, 1)
    recon_img = denormalize(recon_img)
    img = transforms.ToPILImage()(recon_img)
    img.save(output_img)
    #print(f"✅ Reconstructed image saved to {output_img}")

In [6]:
#### ---------------------- VAE defs ---------------------------------------

# Ascon

In [7]:
#pip install ascon

In [8]:
#Libraries for Ascon
import ascon

In [9]:
# --- Configuration ---
# Key: 16 bytes (128-bit)
key_ascon = os.urandom(16)
# Nonce: 16 bytes (recommended)
nonce_ascon = os.urandom(16)
# Associated Data (optional, for authentication metadata)
associated_data_ascon = b"Ascon_LoRa_Experiment"

### Encrypt compressed file and save ciphertext to file

In [10]:
def ascon_encrypt(input_path, key, nonce, associated_data, output_path):
    # Open bin file
    with open(input_path, "rb") as f:
        plaintext = f.read()

    ciphertext = ascon.encrypt(
        key=key,
        nonce=nonce,
        associateddata=associated_data,
        plaintext=plaintext
    )

    # Save encrypted file
    with open(output_path, "wb") as f:
        f.write(ciphertext)

### Decrypt file and save plaintext to output file

In [11]:
def ascon_decrypt(input_encrypt_path, key, nonce, associated_data, output_decrypt_path):
    # Open encrypted file
    with open(input_encrypt_path, "rb") as f:
        ciphertext = f.read()

    plaintext = ascon.decrypt(
        key=key,
        nonce=nonce,
        associateddata=associated_data,
        ciphertext=ciphertext
    )
    
    # Check for authentication failure (wrong key/nonce/AD)
    if plaintext is None:
        print("❌ Decryption failed: authentication error. "
              "Possible causes: incorrect key, nonce, or associated data.\n")
    else:
        # Save decrypted file
        with open(output_decrypt_path, "wb") as f:
            f.write(plaintext)

In [12]:
#### ---------------------- VAE defs ---------------------------------------

# ChaCha20

In [13]:
#pip install cryptography

In [14]:
#Libraries for ChaCha20
#from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives.ciphers.aead import ChaCha20Poly1305
from cryptography.exceptions import InvalidTag

In [15]:
# --- Configuration ---
# Key: 32 bytes (256-bit requirements)
key_chacha20 = os.urandom(32)
# Nonce: 12 bytes (96-bit requirements)
nonce_chacha20 = os.urandom(12)
# Associated Data (if use Poly1305, for authentication metadata)
associated_data_chacha20 = b"ChaCha20_LoRa_Experiment"

### Encrypt compressed file with ChaCha20 and save ciphertext

In [16]:
def chacha20_encrypt(input_path, key, nonce, associated_data, output_path):

    # Read plaintext
    with open(input_path, "rb") as f:
        plaintext = f.read()

    aead = ChaCha20Poly1305(key)
    # encrypt returns ciphertext with 16-byte auth tag appended
    ciphertext = aead.encrypt(nonce=nonce, data=plaintext, associated_data=associated_data)

    # Write ciphertext to file
    with open(output_path, "wb") as f:
        f.write(ciphertext)

### Decrypt compressed file using ChaCha20-Poly1305 and save output

In [17]:
def chacha20_decrypt(input_path, key, nonce, associated_data, output_path):

    # Open encrypted file
    with open(input_path, "rb") as f:
        ciphertext = f.read()

    aead = ChaCha20Poly1305(key)
    
    plaintext = aead.decrypt(nonce=nonce, data=ciphertext, associated_data=associated_data)
    
    # Check for authentication failure (wrong key/nonce/AD)
    if plaintext is None:
        print("❌ Decryption failed: authentication error. "
              "Possible causes: incorrect key, nonce, or associated data.\n")
    else:
        # Save decrypted file
        with open(output_path, "wb") as f:
            f.write(plaintext)

# CRYSTALS-Kyber

In [18]:
#pip install pqcrypto
#pip install pycryptodome

#pip install kyber-py

In [19]:
# Libraries
# References GitHub repo: https://github.com/giacomopope/kyber-py

# PyPI package: kyber-py Library
from kyber_py.kyber import Kyber512, Kyber768, Kyber1024

In [20]:
#Clone and install manually:
#git clone https://github.com/GiacomoPope/kyber-py
#cd kyber-py
#pip install .

#### Receiver generates:

In [21]:
def kyber_generate_keys(level=512):
    """
    Generates a post-quantum public/secret keypair using CRYSTALS-Kyber.

    level: 512, 768, or 1024
    returns: (public_key, secret_key)
    """
    if level == 512:
        K = Kyber512
    elif level == 768:
        K = Kyber768
    elif level == 1024:
        K = Kyber1024
    else:
        raise ValueError("Kyber level must be 512, 768, or 1024")

    public_key, secret_key = K.keygen()
    return public_key, secret_key

In [22]:
def kyber_decapsulate(secret_key, cyphertext_kyber, level=512):
    """
    Decapsulates the KEM ciphertext to recover the shared secret.
    Used on receiver side.
    """
    if level == 512:
        K = Kyber512
    elif level == 768:
        K = Kyber768
    elif level == 1024:
        K = Kyber1024

    key_kyber = K.decaps(secret_key, cyphertext_kyber)
    return key_kyber

#### Transmitter generation key for symmetric algorithms (ASCON or ChaCha20):

In [23]:
def kyber_encapsulate(public_key, level=512):
    """
    Encapsulates a shared secret using the receiver's public key.
    Produces ciphertext_kem (to send over LoRa)
    and shared_secret (local, used to derive ChaCha20 key)
    """
    if level == 512:
        K = Kyber512
    elif level == 768:
        K = Kyber768
    elif level == 1024:
        K = Kyber1024

    key_kyber, cyphertext_kyber = K.encaps(public_key)
    return key_kyber, cyphertext_kyber


# File

In [24]:
# Check file size
def check_file_size(input_path):
    file_size = os.path.getsize(input_path) 
    print(f"The file {input_path} size is: {file_size} bytes") 

In [25]:

# Image
image_path = 'forest.jpg'

# VAE compress
vae_compressed_path = 'vae_compressed.bin'
# VAE decompress
vae_decompress_image_path = "forest_de_compressed.png"

# Ascon encrypted
ascon_encrypted_path = "ascon_encrypted.bin"
# Ascon decrypted
ascon_decrypted_path = "ascon_decrypted.bin"
# Ascon VAE decompress
ascon_decompress_image_path = "forest_ascon_de_compressed.png"

# ChaCha20 encrypted
chacha20_encrypted_path = "chacha20_encrypted.bin"
# ChaCha20 decrypted
chacha20_decrypted_path = "chacha20_decrypted.bin"
# ChaCha20 VAE decompress
chacha20_decompress_image_path = "forest_chacha20_de_compressed.png"

# CRYSTALS-Kyber encrypted
kyber_encrypted_path = "kyber_encrypted.bin"
# ChaCha20 decrypted
kyber_decrypted_path = "kyber_decrypted.bin"
# ChaCha20 VAE decompress
kyber_decompress_image_path = "forest_kyber_de_compressed.png"

#### Image size

In [26]:
check_file_size(image_path)

The file forest.jpg size is: 5683859 bytes


#### VAE implementation

In [27]:
# VAE Compress
compress_image_for_lora(image_path, vae_compressed_path)
check_file_size(vae_compressed_path)

The file vae_compressed.bin size is: 1016 bytes


In [28]:
# VAE Decompress
decompress_image_from_lora(vae_compressed_path, vae_decompress_image_path)
check_file_size(vae_decompress_image_path)

The file forest_de_compressed.png size is: 37818 bytes


In [29]:
###########################################################

#### Ascon implementation

In [30]:
print(f"The Ascon key size is: {len(key_ascon)} bytes") 
print(f"The Ascon key nonce is: {len(nonce_ascon)} bytes") 
print(f"The Ascon key associated data is: {len(associated_data_ascon)} bytes") 

The Ascon key size is: 16 bytes
The Ascon key nonce is: 16 bytes
The Ascon key associated data is: 21 bytes


In [31]:
#Ascon encryption
ascon_encrypt(vae_compressed_path, key_ascon, nonce_ascon, associated_data_ascon, ascon_encrypted_path)
check_file_size(ascon_encrypted_path)

The file ascon_encrypted.bin size is: 1032 bytes


In [32]:
#Ascon decryption
ascon_decrypt(ascon_encrypted_path, key_ascon, nonce_ascon, associated_data_ascon, ascon_decrypted_path)
check_file_size(ascon_decrypted_path)

The file ascon_decrypted.bin size is: 1016 bytes


In [33]:
# VAE Decompress Ascon decrypted file
decompress_image_from_lora(ascon_decrypted_path, ascon_decompress_image_path)
check_file_size(ascon_decompress_image_path)

The file forest_ascon_de_compressed.png size is: 37818 bytes


In [34]:
##################################################################

#### ChaCha10 implementation 

In [35]:
print(f"The ChaCha10 key size is: {len(key_chacha20)} bytes") 
print(f"The ChaCha10 key nonce is: {len(nonce_chacha20)} bytes") 
print(f"The ChaCha10 key associated data is: {len(associated_data_chacha20)} bytes")

The ChaCha10 key size is: 32 bytes
The ChaCha10 key nonce is: 12 bytes
The ChaCha10 key associated data is: 24 bytes


In [36]:
#ChaCha10 encryption
chacha20_encrypt(vae_compressed_path, key_chacha20, nonce_chacha20, associated_data_chacha20, chacha20_encrypted_path)
check_file_size(chacha20_encrypted_path)

The file chacha20_encrypted.bin size is: 1032 bytes


In [37]:
#ChaCha10 decryption
chacha20_decrypt(chacha20_encrypted_path, key_chacha20, nonce_chacha20, associated_data_chacha20, chacha20_decrypted_path)
check_file_size(chacha20_decrypted_path)

The file chacha20_decrypted.bin size is: 1016 bytes


In [38]:
# VAE Decompress ChaCha10 decrypted file
decompress_image_from_lora(chacha20_decrypted_path, chacha20_decompress_image_path)
check_file_size(chacha20_decompress_image_path)

The file forest_chacha20_de_compressed.png size is: 37818 bytes


In [39]:
###############################################################

#### CRYSTALS-Kyber implementation

In [40]:
###### Sizes of public_key and secret_key based on the level 
public_key, secret_key  = kyber_generate_keys(512)
print(f"Level  512: The CRYSTALS-Kyber public key size is: {len(public_key)} bytes")
print(f"Level  512: The CRYSTALS-Kyber secret key size is: {len(secret_key)} bytes")
public_key, secret_key  = kyber_generate_keys(768)
print(f"Level  768: The CRYSTALS-Kyber public key size is: {len(public_key)} bytes")
print(f"Level  768: The CRYSTALS-Kyber secret key size is: {len(secret_key)} bytes")
public_key, secret_key  = kyber_generate_keys(1024)
print(f"Level 1024: The CRYSTALS-Kyber public key size is: {len(public_key)} bytes")
print(f"Level 1024: The CRYSTALS-Kyber secret key size is: {len(secret_key)} bytes")

Level  512: The CRYSTALS-Kyber public key size is: 800 bytes
Level  512: The CRYSTALS-Kyber secret key size is: 1632 bytes
Level  768: The CRYSTALS-Kyber public key size is: 1184 bytes
Level  768: The CRYSTALS-Kyber secret key size is: 2400 bytes
Level 1024: The CRYSTALS-Kyber public key size is: 1568 bytes
Level 1024: The CRYSTALS-Kyber secret key size is: 3168 bytes


In [41]:
# the Receiver generates public and secret key. Bublic key was sent to the Transmitter:
public_key_kyber, secret_key_kyber = kyber_generate_keys(512)

In [42]:
# CRYSTALS-Kyber the Transmitter generates key and cyphertext:
key_kyber, cyphertext_kyber = kyber_encapsulate(public_key_kyber)

In [49]:
print(f"The Kyber key size is: {len(public_key)} key_kyber")
print(f"The CRYSTALS-Kyber cyphertext size is: {len(secret_key)} bytes")

The Kyber key size is: 1568 key_kyber
The CRYSTALS-Kyber cyphertext size is: 3168 bytes


In [43]:
# Transmitter uses ChaCha10 encryption and a encapsulated key using public key from the Receiver
chacha20_encrypt(vae_compressed_path, key_kyber, nonce_chacha20, associated_data_chacha20, kyber_encrypted_path)
check_file_size(kyber_encrypted_path)

The file kyber_encrypted.bin size is: 1032 bytes


In [44]:
# the Transmitter send encrypted file and cyphertext to the Receiver

In [45]:
# the Receiver decapsulates shared secrets:
receiver_key_kyber = kyber_decapsulate(secret_key_kyber, cyphertext_kyber)

In [46]:
# the Receiver uses ChaCha10 decryption and a decapsulated key
chacha20_decrypt(kyber_encrypted_path, receiver_key_kyber, nonce_chacha20, associated_data_chacha20, kyber_decrypted_path)
check_file_size(kyber_decrypted_path)

The file kyber_decrypted.bin size is: 1016 bytes


In [47]:
# VAE Decompress CRYSTALS-Kyber decrypted file
decompress_image_from_lora(kyber_decrypted_path, kyber_decompress_image_path)
check_file_size(kyber_decompress_image_path)

The file forest_kyber_de_compressed.png size is: 37818 bytes


In [48]:
stoppp

NameError: name 'stoppp' is not defined

# ------------------------------- Set up LoRa ------------------------------------

In [ ]:
# Transmitter configuration
# Begin LoRa and set NSS, reset, busy, IRQ, txen, and rxen pin with connected Raspberry Pi gpio pins
busId = 0; csId = 0
resetPin = 18; busyPin = 20; irqPin = 16; txenPin = 6; rxenPin = -1
LoRa = SX126x()

# Initialize LoRa
print("Initializing LoRa")
if not LoRa.begin(busId, csId, resetPin, busyPin, irqPin, txenPin, rxenPin):
    raise Exception("Unable to initialize LoRa")

LoRa.setDio2RfSwitch()

# Set frequency to 868 Mhz, Europe
lora_frequency = 868 * 1000000
LoRa.setFrequency(lora_frequency)

# Set TX power for SX1262 is +22 dBm (https://www.semtech.com/products/wireless-rf/lora-connect/sx1262)
lora_tx_power = 22
LoRa.setTxPower(lora_tx_power, LoRa.TX_POWER_SX1262)

# Configure modulation parameter: spreading factor, bandwidth, and coding rate (Receiver must have same parameters)
lora_spreading_factor = 7                       # LoRa spreading factor
lora_bandwidth = 125 * 1000                     # LoRa Bandwidth: "n" kHz
lora_coding_rate = 5                            # LoRa #Coding rate:  4/5 4/6 4/7 4/8
LoRa.setLoRaModulation(lora_spreading_factor, lora_bandwidth, lora_coding_rate)

# Cofigurate LoRa packet parameter for data transmission
# <<< Explicit >>>
headerType = LoRa.HEADER_EXPLICIT                               # Explicit header mode
preambleLength = 12                                             # Set preamble length to 12
payloadLength = 255                                             # Initialize payloadLength
crcType = True                                                  # Set CRC enable


LoRa.setLoRaPacket(headerType, preambleLength, payloadLength, crcType)


# Set syncronize word for public network (0x3444)
LoRa.setSyncWord(0x3444)

print("\n-- LoRa File Transmitter --\n")

# def generate_random_flag():

In [ ]:
#    #Generate a random 4-character numeric flag.
#    return ''.join(random.choices(string.digits, k=4))

# Compress image using AVIF
def compress_avif(input_path, output_path, quality=100, lossless=False):
    # Start timing
    start_time = time.time()  
   
    # RGBA for True lossless, RGB for lossy
    img = Image.open(input_path).convert("RGBA" if lossless else "RGB")  

    if lossless:
        # True lossless with RGBA
        img.save(output_path, "AVIF", lossless=True, quality=quality)  
    else:
        # Lossy mode with Y'CbCr 4:2:0
        img.save(output_path, "AVIF", lossless=False, quality=quality)
   
    # End timing
    end_time = time.time()
    # Calculate duration
    elapsed_time = end_time - start_time

    print(f"✅ AVIF Compressed: {input_path} -> {output_path}")
    print(f"✅ Compression completed in {elapsed_time:.2f} seconds")
    print(f"✅ Quality =", quality)
    #print_compression_stats(input_path, output_path)

# Create a file name

In [ ]:
def get_next_image_filename(directory):
    #pattern = re.compile(r"(\d{4})_image\.jpg")
    #pattern = re.compile(r"(\d{4})_image\.avif")
    pattern = re.compile(r"(\d{4})_latent_image\.bin")
    max_number = 0

    for filename in os.listdir(directory):
        match = pattern.fullmatch(filename)
        if match:
            number = int(match.group(1))
            max_number = max(max_number, number)

    next_number = max_number + 1
    #return f"{next_number:04d}_image.jpg", f"{next_number:04d}"
    #return f"{next_number:04d}_image.avif", f"{next_number:04d}"
    return f"{next_number:04d}_latent_image.bin", f"{next_number:04d}"

In [ ]:
# Taking image
def take_image(file_path):

    filename = file_path

    width = '3280'
    height = '1200'
    quality = '50'

    cmd = [
        'rpicam-jpeg',
        '--output', filename,
        '--width', width,
        '--height', height,
        '--quality', quality
    ]

    subprocess.run(cmd)

In [ ]:
# Path to the file to transmit using current dir
current_dir, this_filename = os.path.split(__file__)

# Image
image_path = os.path.join(current_dir, 'sending_image.jpg')


image_name, file_flag = get_next_image_filename(current_dir)

#file_path = os.path.join(current_dir, 'image.jpg')
file_path = os.path.join(current_dir, image_name)
file_path_fileANDtime = os.path.join(current_dir, 'stat_file.txt')

In [ ]:

#print("File path = ", file_path)


# BIN file
#file_path = os.path.join(current_dir, 'compressed_vae.bin')
#file_path_fileANDtime = os.path.join(current_dir, 'compressed_vae_bin.txt')

#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# Create a file flag
#file_flag = generate_random_flag()


# Create a file name
#file_flag = get_next_image_filename(current_dir)


# !!!!!!!! working code for taking an image
# Taking image
# image_raw = os.path.join(current_dir, 'image.jpg')
# take_image(image_raw)


# Compress image AVIF
#compress_avif(image_raw, file_path, quality=5)


# Compress image VAE
#image_path = os.path.join(current_dir, 'sending_image.jpg')
#file_path = os.path.join(current_dir, 'latent_lora.bin')


print("File path = ", file_path)
start = time.time()
compress_image_for_lora(image_path, file_path)
end = time.time()


#  Estimate Energy Usage: Compression
duration = end - start
print(f"Compression time: {duration:.2f} seconds")

# Estimate energy if Pi uses 4W average
estimated_power = 5  # Watts
energy = estimated_power * duration
print(f"Estimated Energy: {energy:.2f} joules")





# Record the start time
start_time = time.time()


with open(file_path, "rb") as file:
    chunk_counter = 0
    while chunk := file.read(240):  # 255 bytes max payload minus chunk header (15)
        header = f"{file_flag}|{chunk_counter:05d}|{len(chunk):03d}|"
        payload = header.encode() + chunk
        LoRa.beginPacket()
        LoRa.write(list(payload), len(payload))
        LoRa.endPacket()
        print(f"Sent chunk {chunk_counter}: {chunk}")

        # Debugging prints
        print(f"Header: {header}")
        print(f"Header size: {len(header)}")
        print(f"Chunk size: {len(chunk)}")

        chunk_counter += 1
        time.sleep(1)  # Delay between chunks

#Record the end time
end_time = time.time()

#Calculate transmission time
transmission_time = end_time - start_time

    #with open(f"{file_path_image}.txt", "a") as txt:
    #    txt.write(f"{file_flag}: send {chunk_counter} chunks in {transmission_time:.2f} seconds\n")

f = open(file_path_fileANDtime, "a")
f.write(f"{file_flag}: send {chunk_counter} chunks in {transmission_time:.2f} seconds Compression time: {duration:.2f} seconds Estimated Energy: {energy:.2f} Joules\n")
f.close()

print(f"File transmission complete in {transmission_time:.2f} seconds")


LoRa.end()
tx_image_file_EXPLICIT.py
Displaying tx_image_file_EXPLICIT.py.